In [ ]:
from datetime import datetime
# from opera_utils import disp
import geopandas as gpd
import os
import sys
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re

from opera_utils.download import L2Product
from transboundary_opera import run1_download_DISP_S1_Static #, run2_prep_mintpy_opera

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: Maybe look at making this a pixi task? Could make it easier to run?


In [ ]:
SEARCH_START = datetime(2024, 6, 1)     # next runs should start at Jan 1, 2016
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")     # this only gets one set of frame ids 

In [ ]:
# Initialize a list to store frame IDs for each row
all_frame_ids = []

for entry, row in gdf.iterrows():
    
    results = asf.geo_search(
        intersectsWith=row.geometry.convex_hull.wkt,
        dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.DISP_S1,
        maxResults=50  # We only need a few results to identify the Frame IDs
    )
    
    # Set for the current row
    current_frame_ids = set()
    pattern = re.compile(r'_F(\d{5})_')

    for product in results:
        filename = product.properties['fileName']
        match = pattern.search(filename)
        if match:
            frame_id_str = match.group(1) 
            current_frame_ids.add(int(frame_id_str))
    
    # Add the sorted list of frame IDs to our collection
    all_frame_ids.append(sorted(list(current_frame_ids)))

# Assign the collected frame IDs to a new column in the GeoDataFrame
gdf['frame_ids'] = all_frame_ids

unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})
len(unique_frame_ids)

In [ ]:
unique_frame_ids = [1092,1093,1094,1095,1096,1097,3065,3066,3067,5124,5125,7091,7092,7093,14877,14878,14879,16939,16940,18905,20694,20695,20696,20697,22665,22666,22667,24724,24725,26691,26692,28478,28479,28480,28481,28482,30452,30453,30454,30455,34477,34478,34479,38241,40295,40296,40297,40298,42264,42265,44325,44326,46290,46291]

## Download OPERA datasets that report LOS displacement. Should process into .h5 format for input into mintpy, but does not include geometry for some reason.
- Geometry will be generated using run1_download_DISP_S1_Static. This function generates los_east.tif and los_north.tif. These tifs will be used to determine the azimuth and incidence angles needed for decomposing the LOS displacement

In [ ]:
# path to mintpy bash script
script_path = Path(os.getcwd()).parent / 'create-mintpy_cc.sh'

for frame_id in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame_id)
    os.makedirs(out_path, exist_ok=True)

    if (out_path / "mintpy").exists():
        print(f"Frame ID {frame_id} already processed. Skipping.")
        continue
    
    else:
        # 2. Define the REQUIRED environment variables for the script
        required_env = {
            "FRAME_ID": f"{frame_id}",  # Using the current frame ID from the loop
            "START": SEARCH_START.strftime("%Y-%m-%d"),
            "END": SEARCH_END.strftime("%Y-%m-%d"),
            }

        # 3. Define OPTIONAL environment variables to override the script's defaults
        optional_env = {
            "NUM_WORKERS": "8",              # Overrides default of 4
            "REFERENCE_METHOD": "HIGH_COHERENCE"    # Overrides default of BORDER. Either HIGH_COHERENCE or BORDER would be best I think.
        }
        print(f"Preparing to execute script: {script_path}")
        print(f"With required environment: {required_env}")

        # Construct the full environment: current system environment + custom variables
        full_environment = os.environ.copy()
        full_environment.update(required_env)
        full_environment.update(optional_env)

        try:
            # Execute the script directly (preferred over shell=True)
            result = subprocess.run(
                [script_path],
                check=True,  # Raise CalledProcessError for non-zero exit codes
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,  # Decode output as string
                env=full_environment,
                cwd = out_path
            )

            # Success output
            print("\nScript Execution Successful")
            print(f"Return Code: {result.returncode}")
            print("\nSTDOUT")
            print(result.stdout)
            if result.stderr:
                print("\nSTDERR (Non-fatal messages)")
                print(result.stderr)
        except subprocess.CalledProcessError as e:
            # Error handling for script failure (e.g., if a command inside the script fails)
            print(f"\nERROR: Script Failed (Exit Code {e.returncode})")
            print(f"Command run: {e.cmd}")
            print("\nSTDOUT (Captured before failure)")
            print(e.stdout)
            print("\nSTDERR (Error messages)")
            print(e.stderr)

        except FileNotFoundError:
            # Error handling for when the script file itself cannot be found
            print(f"\nCRITICAL ERROR: Script file not found or lacks execute permission at: {script_path}")

### Use run1_download_DISP_S1_Static to download CLSC Static files needed for orbit geometry

In [ ]:
for frame in unique_frame_ids:
    staticDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'orbit_data'
    os.makedirs(staticDir, exist_ok=True)
    dispDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'disp_data'
    os.makedirs(dispDir, exist_ok=True)
    geomDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame) / 'geom_data'
    os.makedirs(geomDir, exist_ok=True)

    sys.argv = [
        'notebook',  # placeholder for script name
        '--frameID', str(frame),
        '--startDate', SEARCH_START.strftime('%Y%m%d'),
        '--endDate', SEARCH_END.strftime('%Y%m%d'),
        '--dispDir', str(dispDir),
        '--staticDir', str(staticDir),
        '--geomDir', str(geomDir),
        '--staticOnly'                             # default is true, uncomment for false
    ]

    run1_inps = run1_download_DISP_S1_Static.createParser()
    run1_download_DISP_S1_Static.main(run1_inps)

    if os.path.exists(geomDir.parent / 'mintpy'):
        dt.create_geom_h5_with_ref(geomDir / 'los_east.tif', geomDir / 'los_north.tif', geomDir.parent / 'mintpy' / 'timeseries.h5')
    

## Trying out a different approach for downloading and preparing the data:
1. run /src/transboundary_opera/run1_download_DISP_S1_Static.py to download the OPERA DISP (displacement time series) and CSLC (orbit geometry) products
2. run /src/transboundary_opera/run2_prep_mintpy_opera.py

In [ ]:
sys.argv = [
    "notebook", # placeholder for script name
    "--unw-file-glob",
    "--geom-dir",
    "--meta-file", 
    "--start-date", SEARCH_START.strftime('%Y%m%d'),    # start of search window for OPERA products; any ifgs earlier will be removed
    "--end-date", SEARCH_END.strftime('%Y%m%d'),        # end of search window for OPERA products; any ifgs later will be removed
    "--out-dir",
    "--range", 1,
    "--azimuth", 1, 
    "--water-mask-file", 
    "--dem-file", 'glo_30',                             # can be path to valid DEM or download one of ssrtm_v3, nasadem, glo_30, glo_90, glo_90_missing
    "--ref-lalo", None,                 # 'latitude longitude' of reference point, or None to select highest coherence point by default
    "--zero-mask", True,                # masks all pixels with zero in the unwrapped phase
    "--corr-lyrs", True,                # extract correction layers
    "--shortwvl-lyrs", True,            # extract short wavelength layers
    "--tropo-correction", True,         # apply tropospheric correction using HRRR weather model
    "--work-dir",                       # Working directory for tropospheric correction intermediates
    "--mask-layers", True,              # extract all mask layers
    "--load-all-layers", True,          # extract all layers
    "--apply-mask", True,               # apply epoch based masking
    "--dem-error", True,                # apply dem error correction
    "--n-workers", 6,                   # default is none; number of workers for writing timeseries in parallel
    "--reliability-threshold", 0.9,     # default of 0.9; 
    "--chunk-size", 50,                 # default of 50
]

run2_inps = run2_prep_mintpy_opera.createParser()
run2_prep_mintpy_opera.main(run2_inps)

In [ ]:
import rasterio as rio

test_east = rio.open(test_geomDir / 'los_east.tif')
test_north = rio.open(test_geomDir / 'los_north.tif')
test_layover = rio.open(test_geomDir / 'layover_shadow_mask.tif')

In [ ]:
los_east = test_east.read(1).flatten()
los_north = test_north.read(1).flatten()

In [ ]:
los_east.flatten()
los_north.flatten()

In [ ]:
plt.imshow(test_layover.read(1))

In [ ]:
run1_download_DISP_S1_Static(frameID = te)

In [ ]:
static_products = {}
product = L2Product.CSLC_STATIC

for frame in unique_frame_ids:
    static_products[frame] = asf.search(
        frame=frame,
        # dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.CSLC_STATIC,
    )

In [ ]:
static_products

In [ ]:
# search CLSC Static Layer files
product = L2Product.CSLC_STATIC

results = asf.search(
    frame=test_frame,
    processingLevel=product.value,
)

# results.download(path=test_staticDir, processes=5)    # downloading static layers with simultaneous downloads